# Assembly101 — operator variation

**This notebook runs top to bottom whether or not you have dataset access yet.**

If the Hugging Face gate is still closed it falls back to a structurally identical synthetic
stand-in: same 48 participants, same recording names, same long-tailed verb distribution. Every
figure produced that way is stamped `SYNTHETIC — NOT A RESULT` so it can never be mistaken for
one.

When access is granted, change nothing. Re-run. It picks up the real data automatically and the
stamp disappears.

## 1. Setup

In [ ]:
!pip install -q huggingface_hub lmdb scikit-learn pandas matplotlib

import os, re, collections
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import GroupKFold, KFold
from sklearn.metrics import f1_score

REPO, N_VERBS, SYNTH_DIM = "cvml-nus/assembly101", 24, 256
DATA = "/content/a101"; os.makedirs(DATA, exist_ok=True)
REC_RE = re.compile(r"nusar-\d{4}_action_both_(\d+)-")

def participant_from_recording(name):
    m = REC_RE.match(str(name)); return m.group(1) if m else None

## 2. Access diagnostic

Tells you exactly which of three states you are in, instead of a stack trace you have to decode.
Safe to run with no token.

In [ ]:
def check_access():
    from huggingface_hub import whoami, hf_hub_download
    try:
        user = whoami()["name"]
    except Exception:
        print("STATE 1/3 — not authenticated.")
        print("  Fix: huggingface.co/settings/tokens -> New token -> type Read.")
        print("  Then put it in Colab Secrets as HF_TOKEN (key icon, left sidebar).")
        return False
    print(f"authenticated as: {user}")
    try:
        hf_hub_download(REPO, "annotations/README.md", repo_type="dataset", local_dir=DATA)
        print("STATE 3/3 — access granted. Real data will be used.")
        return True
    except Exception as e:
        if "403" in str(e) or "not in the authorized list" in str(e):
            print(f"STATE 2/3 — token works, but '{user}' is not authorized yet.")
            print(f"  Fix: open huggingface.co/datasets/{REPO} logged in AS {user}")
            print("       and accept the agreement. If it queues, wait for approval.")
        else:
            print(f"unexpected: {type(e).__name__}: {e}")
        return False

try:
    from google.colab import userdata
    from huggingface_hub import login
    tok = userdata.get('HF_TOKEN')
    if tok: login(token=tok, add_to_git_credential=False)
except Exception:
    pass

HAVE_ACCESS = check_access()

## 3. Participant structure

Public file, no auth, no gate. This is the check that decides the project is possible, and it
works today regardless of everything above.

In [ ]:
!wget -q -O recording_names.txt https://raw.githubusercontent.com/assembly-101/assembly101-download-scripts/main/recording_names.txt

ALL = [l.strip() for l in open("recording_names.txt") if l.strip()]

# The name carries the participant id TWICE. In 7 of 362 they disagree, so we
# cannot just take the first and hope. Rule: if the second token never appears
# as a first token anywhere, it is a harmless alias (same group either way).
# If BOTH tokens are real participant groups, the assignment is genuinely
# ambiguous and we drop that recording rather than guess -- misassigning one
# person's footage to another is precisely the leak this project measures.
PAT = re.compile(r"nusar-\d{4}_action_both_(\d+)-([a-z0-9]+)_(\d+)_user_id_")
_p = {l: PAT.match(l).groups() for l in ALL}
_firsts = {g[0] for g in _p.values()}
AMBIGUOUS = [l for l, (a, t, b) in _p.items() if a != b and b in _firsts]
ALIASED   = [l for l, (a, t, b) in _p.items() if a != b and b not in _firsts]
print(f"benign aliases     {len(ALIASED)}  (same grouping either way)")
print(f"ambiguous, dropped {len(AMBIGUOUS)}  ({100*len(AMBIGUOUS)/len(ALL):.2f}% of recordings)")
for l in AMBIGUOUS: print("   ", l)
print()

RECS = [l for l in ALL if l not in set(AMBIGUOUS)]
pids = [participant_from_recording(r) for r in RECS]
assert all(pids), "some recording names failed to parse"
cnt = collections.Counter(pids); v = np.array(list(cnt.values()))

print(f"recordings        {len(RECS)}")
print(f"participants      {len(cnt)}")
print(f"recordings/person min {v.min()}  median {np.median(v)}  max {v.max()}")
print(f"\nSession ids would number ~{len(RECS)}. There are {len(cnt)}, each with {v.min()}+ recordings.")
print("These are people. Leave-one-participant-out is viable.")

## 4. Segments — real if available, synthetic otherwise

`harmonise()` maps whatever the real CSV calls its columns onto four canonical names, trying
several known aliases, and fails loudly listing the actual columns if none match. That is the
cell most likely to need a one-line edit when real data arrives.

In [ ]:
COL_GUESS = {
    "recording": ["video", "video_id", "recording", "video_name"],
    "verb":      ["verb_id", "verb", "verb_cls"],
    "start":     ["start_frame", "start", "action_start_frame"],
    "end":       ["end_frame", "end", "action_end_frame"],
}

def harmonise(raw):
    ren, missing = {}, []
    for want, opts in COL_GUESS.items():
        hit = next((c for c in opts if c in raw.columns), None)
        if hit: ren[hit] = want
        else: missing.append(want)
    if missing:
        raise KeyError(f"no column for {missing}. Available: {list(raw.columns)}")
    df = raw.rename(columns=ren)[list(COL_GUESS)].copy()
    df["participant"] = df["recording"].map(participant_from_recording)
    bad = df["participant"].isna().sum()
    if bad: print(f"  warning: dropping {bad} unparseable rows")
    return df.dropna(subset=["participant"]).reset_index(drop=True)

def synthetic_segments(seed=0):
    rng = np.random.default_rng(seed)
    prior = rng.dirichlet(np.ones(N_VERBS) * 0.4)
    pref, rows = {}, []
    for rec in RECS:
        p = participant_from_recording(rec)
        pref.setdefault(p, rng.dirichlet(prior * 40 + 0.2))
        frame = int(rng.integers(0, 600))
        for _ in range(int(rng.integers(120, 260))):
            v = int(rng.choice(N_VERBS, p=pref[p]))
            dur = int(max(6, rng.normal(50 + 18 * (int(p) % 7), 14)))
            rows.append((rec, p, v, frame, frame + dur)); frame += dur + int(rng.integers(2, 25))
    return pd.DataFrame(rows, columns=["recording", "participant", "verb", "start", "end"])

def load_segments():
    if HAVE_ACCESS:
        from huggingface_hub import hf_hub_download
        parts = []
        for s in ["train", "validation"]:
            p = hf_hub_download(REPO, f"annotations/fine-grained-annotations/{s}.csv",
                                repo_type="dataset", local_dir=DATA)
            parts.append(pd.read_csv(p))
        return harmonise(pd.concat(parts, ignore_index=True)), "REAL"
    for c in ["train.csv", f"{DATA}/train.csv"]:      # manual upload path
        if os.path.exists(c):
            print(f"  using uploaded {c}")
            return harmonise(pd.read_csv(c)), "REAL"
    return synthetic_segments(), "SYNTHETIC"

df, MODE = load_segments()
df["n_frames"] = df["end"] - df["start"]
print(f"\nMODE = {MODE}")
print(f"segments {len(df):,}   participants {df['participant'].nunique()}   verbs {df['verb'].nunique()}")
df.head()

## 5. Rung 1 — variation visible from labels alone

No features, no GPU. If the gate opens late, **this is still a real result you can present.**

In [ ]:
def stamp(fig):
    if MODE == "SYNTHETIC":
        fig.text(.5, .5, "SYNTHETIC — NOT A RESULT", fontsize=34, color="red",
                 alpha=.22, ha="center", va="center", rotation=24, weight="bold")

fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
med = df.groupby("participant")["n_frames"].median().sort_values() / 30.0
ax[0].bar(range(len(med)), med.values, color="#4C72B0")
ax[0].axhline(med.median(), color="crimson", ls="--", lw=1, label=f"median {med.median():.2f}s")
ax[0].set(title="Median segment duration by participant",
          xlabel="participant (sorted)", ylabel="seconds"); ax[0].legend()

piv  = pd.crosstab(df["participant"], df["verb"], normalize="index")
glob = df["verb"].value_counts(normalize=True).reindex(piv.columns).fillna(0).values
def js(p, q, eps=1e-12):
    p, q = p+eps, q+eps; m = .5*(p+q)
    kl = lambda a,b: np.sum(a*np.log(a/b)); return .5*kl(p,m)+.5*kl(q,m)
d = pd.Series([js(piv.loc[i].values, glob) for i in piv.index], index=piv.index).sort_values()
ax[1].bar(range(len(d)), d.values, color="#DD8452")
ax[1].set(title="Divergence of verb mix from the population",
          xlabel="participant (sorted)", ylabel="Jensen-Shannon divergence")
stamp(fig); plt.tight_layout(); plt.savefig("fig_label_variation.png", dpi=150); plt.show()

print(f"slowest / fastest median duration: {med.max()/med.min():.2f}x")
print(f"most idiosyncratic verb mix: participant {d.index[-1]}")

## 6. Features

Real mode inspects the lmdb key format before reading anything, because I am not guessing it.
Synthetic mode generates features with a shared class signal plus a persistent per-operator
offset, which is the effect the harness exists to detect.

In [ ]:
def peek_lmdb(path, n=5):
    import lmdb
    env = lmdb.open(path, readonly=True, lock=False, subdir=os.path.isdir(path))
    with env.begin() as txn:
        for i, (k, val) in enumerate(txn.cursor()):
            if i >= n: break
            print(f"key={k.decode()!r}  bytes={len(val)}  f32={np.frombuffer(val, np.float32).shape}")

KEY_FMT, MAX_FRAMES, FEAT_DIM = lambda rec, f: f"{rec}/{f:010d}".encode(), 16, 2048

def pool_real(df, lmdb_path):
    import lmdb
    env = lmdb.open(lmdb_path, readonly=True, lock=False, subdir=os.path.isdir(lmdb_path))
    X, keep = [], []
    with env.begin() as txn:
        for i, r in enumerate(df.itertuples()):
            idx = np.linspace(r.start, max(r.start, r.end-1),
                              min(MAX_FRAMES, max(1, r.end-r.start))).astype(int)
            got = [np.frombuffer(b, np.float32) for b in
                   (txn.get(KEY_FMT(r.recording, f)) for f in idx) if b is not None]
            if got and got[0].shape[0] == FEAT_DIM:
                X.append(np.mean(got, 0)); keep.append(i)
            if i % 5000 == 0: print(f"  {i}/{len(df)}", end="\r")
    return np.vstack(X), df.iloc[keep].reset_index(drop=True)

def synthetic_features(df, seed=1):
    rng = np.random.default_rng(seed)
    proto = rng.normal(0, .16, (N_VERBS, SYNTH_DIM))
    styles = {p: rng.normal(0, 1.10, SYNTH_DIM) for p in df["participant"].unique()}
    S = np.stack([styles[p] for p in df["participant"].values])
    return proto[df["verb"].values] + S + rng.normal(0, .95, (len(df), SYNTH_DIM))

if MODE == "SYNTHETIC":
    X, dfk = synthetic_features(df), df
else:
    LMDB_PATH = ""   # <-- set after peek_lmdb(); then: X, dfk = pool_real(df, LMDB_PATH)
    raise SystemExit("Run peek_lmdb() on one downloaded view, set LMDB_PATH and KEY_FMT.")

print(f"X = {X.shape}")

## 7. Harness, and a positive control

Two evaluations identical in every respect except group awareness: same model, same eight folds,
same test sizes. The control checks the harness can see an effect that is definitely present.
If it cannot, nothing it says about real data means anything.

In [ ]:
def make_model(): return make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))

def _fit(X, y, tr, te):
    m = make_model(); m.fit(X[tr], y[tr]); return m.predict(X[te]), y[te]

def evaluate_random(X, y, n_splits=8, seed=0):
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed); P, T = [], []
    for tr, te in kf.split(X):
        p, t = _fit(X, y, tr, te); P.append(p); T.append(t)
    return f1_score(np.concatenate(T), np.concatenate(P), average="macro",
                    labels=np.unique(y), zero_division=0)

def evaluate_grouped(X, y, groups, n_splits=8):
    gkf = GroupKFold(n_splits=n_splits); per, P, T = {}, [], []
    for tr, te in gkf.split(X, y, groups):
        p, t = _fit(X, y, tr, te); P.append(p); T.append(t)
        held = groups[te]
        for q in np.unique(held):
            m = held == q
            # Only the verbs this person performs. Averaging over all 24 would
            # charge them for classes absent from their ground truth, measuring
            # repertoire breadth rather than model quality on that person.
            per[q] = f1_score(t[m], p[m], average="macro",
                              labels=np.unique(t[m]), zero_division=0)
    return f1_score(np.concatenate(T), np.concatenate(P), average="macro",
                    labels=np.unique(y), zero_division=0), per

def spread_stats(per):
    s = pd.Series(per).sort_values()
    return dict(n=len(s), mean=s.mean(), std=s.std(), worst=s.min(),
                worst_id=s.index[0], best=s.max(), best_id=s.index[-1],
                spread=s.max()-s.min())

# positive control on a small planted effect
_r = np.random.default_rng(0)
_p = _r.normal(0,.30,(24,64)); _X,_y,_g = [],[],[]
for i in range(48):
    st_ = _r.normal(0,1.10,64); pr = _r.dirichlet(np.ones(24)*.5)
    lb = _r.choice(24,220,p=pr)
    _X.append(_p[lb]+st_+_r.normal(0,.55,(220,64))); _y.append(lb); _g.append(np.full(220,f"90{i:02d}"))
_X,_y,_g = np.vstack(_X),np.concatenate(_y),np.concatenate(_g)
_ra = evaluate_random(_X,_y); _gr,_pp = evaluate_grouped(_X,_y,_g)
print(f"control: random {_ra:.3f} | grouped {_gr:.3f} | gap {_ra-_gr:.3f}")
assert _ra > _gr and len(_pp) == 48
print("PASS — harness detects a planted operator effect and scores everyone.")

## 8. Rung 2 — the headline

In [ ]:
y, groups = dfk["verb"].values, dfk["participant"].values
rand = evaluate_random(X, y)
grp, per = evaluate_grouped(X, y, groups)

print(f"[{MODE}]")
print(f"random segment split    macro-F1 = {rand:.3f}")
print(f"participants held out   macro-F1 = {grp:.3f}")
print(f"gap                              = {rand-grp:.3f}")

## 9. Rung 3 — the picture

In [ ]:
st = spread_stats(per)
for k, v in st.items():
    print(f"{k:9s} {v:.3f}" if isinstance(v, float) else f"{k:9s} {v}")

s = pd.Series(per).sort_values()
fig, ax = plt.subplots(figsize=(11, 4.5))
ax.bar(range(len(s)), s.values, color="#4C72B0")
ax.axhline(s.mean(), color="crimson", ls="--", lw=1.2, label=f"mean {s.mean():.3f}")
ax.set(title="Macro-F1 per held-out participant — flat means it generalises across people",
       xlabel="held-out participant (sorted)", ylabel="macro-F1")
ax.set_xticks(range(len(s))); ax.set_xticklabels(s.index, rotation=90, fontsize=7)
ax.legend(); stamp(fig)
plt.tight_layout(); plt.savefig("fig_per_participant.png", dpi=150); plt.show()
print(f"\nWorst {st['worst_id']} ({st['worst']:.3f}), best {st['best_id']} ({st['best']:.3f}).")

## 10. Rung 4 — go and watch

Only meaningful in REAL mode. The deliverable is half a page of prose about what that person does
differently, not more code.

In [ ]:
w = dfk[dfk["participant"] == st["worst_id"]]
print(f"participant {st['worst_id']}: {len(w)} segments, {w['recording'].nunique()} recordings\n")
for r_ in sorted(w["recording"].unique()): print("  ", r_)
print(f"\nmedian segment {w['n_frames'].median()/30:.2f}s vs population {dfk['n_frames'].median()/30:.2f}s")
delta = (w["verb"].value_counts(normalize=True) -
         dfk["verb"].value_counts(normalize=True)).dropna().sort_values()
print("\nverb mix vs population:"); print(pd.concat([delta.head(4), delta.tail(4)]))
if MODE == "SYNTHETIC":
    print("\n[SYNTHETIC — these recordings are real names but the labels are not. Do not watch yet.]")

---
### Before you report anything

Check `MODE`. If it says `SYNTHETIC`, you have a working pipeline and **no result**. That is a
fine place to be, and worth saying plainly: the measurement is built and validated, the dataset
gate is pending.

## 11. Plan B — Breakfast (no gate, no permission)

Same harness, different loader. Breakfast has 52 subjects with roughly 33 videos each, against
Assembly101's 48 with 6, so the per-subject score is *more* stable here, not less.

Upload `breakfast_loader.py` alongside this notebook, then run the cells below. The MS-TCN bundle
is public: features, frame-wise labels, and the official subject-disjoint folds.

In [ ]:
# Download Breakfast only -- do NOT pull all three datasets (~30GB total).
# Get the link from https://github.com/yabufarha/ms-tcn (README, "Download the data").
# Expected layout after extracting:
#   data/breakfast/{features/*.npy, groundTruth/*.txt, mapping.txt, splits/*.bundle}

import shutil
print("free disk:", shutil.disk_usage("/content").free / 1e9, "GB")

BF_ROOT = "/content/data/breakfast"   # <-- set after extracting

In [ ]:
from breakfast_loader import load_dataset, subject_from_video

# sanity-check the subject parser against real filenames before trusting it
import glob, os
vids = [os.path.basename(p)[:-4] for p in glob.glob(f"{BF_ROOT}/groundTruth/*.txt")[:5]]
for v in vids:
    print(f"{v:40s} -> {subject_from_video(v)}")

In [ ]:
X, dfk = load_dataset(BF_ROOT)
MODE = "REAL"
y, groups = dfk["verb"].values, dfk["participant"].values

rand = evaluate_random(X, y)
grp, per = evaluate_grouped(X, y, groups)
print(f"\nrandom segment split    macro-F1 = {rand:.3f}")
print(f"subjects held out       macro-F1 = {grp:.3f}")
print(f"gap                              = {rand-grp:.3f}")

st = spread_stats(per)
print(f"\nper-subject: mean {st['mean']:.3f}  spread {st['spread']:.3f}")
print(f"worst {st['worst_id']} ({st['worst']:.3f})  best {st['best_id']} ({st['best']:.3f})")

Then re-run cells 9 and 10 as written — the plot and the worst-subject report are dataset-agnostic.

**Note for the write-up:** Breakfast's official protocol is already subject-disjoint 4-fold CV.
Assembly101's official splits are *toy*-disjoint instead. So the assembly benchmark controls for
new objects and says nothing about new operators, while the cooking benchmark does the opposite.
Worth one line in the README.